In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_json, from_json, current_timestamp, get_json_object, explode, first, explode_outer
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType
import os

In [ ]:
try:
    spark.stop()
except:
    pass

In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.master("local[*]")
.getOrCreate())

In [ ]:
bronze_path = "../data_lake/bronze/batch_data/"
silver_base_path = "../data_lake/silver/"

In [ ]:
bronze_df = (spark.read.parquet(bronze_path)
).withColumnRenamed("fullUrl","id")

resource_types_rows = (bronze_df.select("resource_type")
             .distinct()
             .filter(col("resource_type").isNotNull()).collect()
)
resource_types = [row.resource_type for row in resource_types_rows]

In [ ]:
bronze_df.schema

In [ ]:
df = spark.read.parquet(bronze_path+"resource_type=Patient")
df.schema

In [ ]:
def process_bronze_to_silver(resource_type):
    input_path = os.path.join(bronze_path, f"resource_type={resource_type}/")
    # print(input_path)
    output_path = os.path.join(silver_base_path, "silver_"+resource_type)
    
    df_bronze_filtered = spark.read.parquet(input_path)

    df_pivot = (df_bronze_filtered.select(
    col("fullUrl"),
    explode_outer(col("resource")).alias("key","value"))
    )

    df_pivot_2 = df_pivot.groupBy("fullUrl").pivot("key").agg(first("value")).withColumn("silver_timestamp", current_timestamp())
    # print(df_pivot_2.schema, df_bronze_filtered.schema)

    df_silver = (df_bronze_filtered.alias("a").join(df_pivot_2.alias("p"), col("a.fullUrl") == col("p.fullUrl"), how="left")
             .select(
                 col("a.bundle_resource_type"),
                    col("a.bundle_type"),
                    col("a.request"),
                    col("p.*"),
                    col("a.input_file_name")
             )
             )
    df_silver.write.mode("append").option("mergeSchema", "true").parquet(output_path)

In [ ]:
for resource_type in resource_types:
    process_bronze_to_silver(resource_type)

In [ ]:
try:
    spark.stop()
except:
    pass